# 🎬 **MovieLens: отчёт по проекту**

Перед нами - датасет **MovieLens ml-latest-small**, в котором мы использовали 1000 строк из каждого csv-файла.

Импорт осуществляется из скрипта `movielens_analysis` из нашего модуля.

## Импорт модуля

In [1]:
%timeit -n 1 -r 1 import movielens_analysis
import movielens_analysis

59.6 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Импорт и создание классов

Основная часть работы - создаём все классы из csv файлов, которые были нам даны.

In [2]:
%timeit -n 1 -r 1 movielens_analysis.Movies('../datasets/movies.csv')
movies = movielens_analysis.Movies('../datasets/movies.csv')
ratings = movielens_analysis.Ratings('../datasets/ratings.csv', '../datasets/movies.csv')
users = movielens_analysis.Ratings.Users('../datasets/ratings.csv', '../datasets/movies.csv')
tags = movielens_analysis.Tags('../datasets/tags.csv')
links = movielens_analysis.Links('../datasets/links.csv', '../datasets/movies.csv')

9.64 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [3]:
%timeit -n 1 -r 1 movies.dist_by_release()
release_dist = movies.dist_by_release()

print(f'  Всего уникальных лет в срезе: {len(release_dist)}')
print()
for year, cnt in list(release_dist.items())[:5]:
    bar = '█' * (cnt // 10)
    print(f'  {year}: {bar} ({cnt})')

208 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
  Всего уникальных лет в срезе: 67

  1995: ██████████████████████ (224)
  1994: ██████████████████ (184)
  1996: ██████████████████ (181)
  1993: ██████████ (101)
  1992: ██ (23)


**Вывод:**  
Импортировав все файлы мы можем увидеть, что в первых 1000 строках датасета больше всего фильмов приходится на период `середины 90-х`, которая доминирует в маленькой выборке.

## Проверка на ошибки

Передаём каждому классу файл с неправильной структурой для обработки исключений.

*В путь `Movies` передаём `ratings.csv`*

In [4]:
%timeit -n 1 -r 1 movielens_analysis.Movies('../datasets/movies.csv')
movies_error = None

try:
    wrong_movies = movielens_analysis.Movies('../datasets/ratings.csv')
    movies_error = wrong_movies.dist_by_genres()
except Exception as error:
    movies_error = error
movies_error

8.07 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


ValueError('wrong file structure: ../datasets/ratings.csv')

*В путь `Ratings` передаём `movies.csv`*

In [5]:
%timeit -n 1 -r 1 movielens_analysis.Ratings('../datasets/ratings.csv', '../datasets/movies.csv')
ratings_error = None

try:
    wrong_ratings = movielens_analysis.Ratings('../datasets/movies.csv', '../datasets/movies.csv')
    ratings_error = wrong_ratings.top_by_num_of_ratings(3)
except Exception as error:
    ratings_error = error
ratings_error

32.6 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


ValueError('wrong file structure: ../datasets/movies.csv')

*В путь `Tags` передаём `movies.csv`*

In [6]:
%timeit -n 1 -r 1 movielens_analysis.Tags('../datasets/tags.csv')
tags_error = None

try:
    wrong_tags = movielens_analysis.Tags('../datasets/movies.csv')
    tags_error = wrong_tags.most_popular(3)
except Exception as error:
    tags_error = error
tags_error

18.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


ValueError('wrong file structure: ../datasets/movies.csv')

*В путь `Links` передаём `movies.csv`*

In [7]:
%timeit -n 1 -r 1 movielens_analysis.Links('../datasets/links.csv', '../datasets/movies.csv')
links_error = None

try:
    wrong_links = movielens_analysis.Links('../datasets/movies.csv', '../datasets/movies.csv')
    links_error = wrong_links.missing_tmdb(3)
except Exception as error:
    links_error = error
links_error

8.78 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


ValueError('wrong file structure: ../datasets/movies.csv')

**Вывод:**  
Все четыре класса корректно отработали ошибки на импорт неправильных файлов.

# **Основная часть**

## *Вопрос 1: Какие годы сильнее всего представлены в этой части датасета? А также какие жанры?*

В начале отчёта мы видим, что середина 90-х доминирует в маленькой выборке. Выбрав срез побольше, мы видим, что пик выпуска фильмов приходится на `1993-1996 годы`, где пишется мини-история в индустрии кино – эпоха голливудских блокбастеров.

In [8]:
%timeit -n 1 -r 1 movies.dist_by_release()
release_dist = movies.dist_by_release()

print(f'  Всего уникальных лет в срезе: {len(release_dist)}')
print()
for year, cnt in list(release_dist.items())[:10]:
    bar = '█' * (cnt // 3)
    print(f'  {year}: {bar} ({cnt})')

175 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
  Всего уникальных лет в срезе: 67

  1995: ██████████████████████████████████████████████████████████████████████████ (224)
  1994: █████████████████████████████████████████████████████████████ (184)
  1996: ████████████████████████████████████████████████████████████ (181)
  1993: █████████████████████████████████ (101)
  1992: ███████ (23)
  1990: █████ (15)
  1991: █████ (15)
  1989: ████ (14)
  1986: ███ (9)
  1940: ██ (8)


В первую очередь зритель смотрит не только на актёрский состав фильма, но и на его `жанр`.  
Какой жанр доминирует в нашем датасете - мы увидим ниже.  

Мы добавили *Unicode bar*, чтобы визуализировать данные для наилучшего восприятия.

In [9]:
%timeit -n 1 -r 1 movies.dist_by_genres()

movie_by_genre = movies.dist_by_genres()
print('Распределение фильмов по жанрам:')
max_count = list(movie_by_genre.values())[0]
for genre, cnt in movie_by_genre.items():
    bar = '█' * int(cnt / max_count * 30)
    print(f' {genre:<22} {bar} {cnt}')

372 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Распределение фильмов по жанрам:
 Drama                  ██████████████████████████████ 507
 Comedy                 █████████████████████ 365
 Romance                ████████████ 208
 Thriller               ██████████ 179
 Action                 █████████ 158
 Adventure              ███████ 126
 Crime                  ███████ 122
 Children               █████ 100
 Fantasy                ████ 69
 Sci-Fi                 ████ 69
 Mystery                ███ 58
 Musical                ███ 53
 Horror                 ███ 51
 War                    ██ 48
 Animation              ██ 37
 Documentary            █ 25
 Western                █ 23
 Film-Noir              █ 18
 IMAX                    3


In [10]:
%timeit -n 1 -r 1 movies.most_genres(10)
multi_genre = movies.most_genres(10)

print('Фильмы с наибольшим количеством жанров (топ-10):')
for i, (title, cnt) in enumerate(multi_genre.items(), 1):
    print(f' {i:>2}. [{cnt} жанра/ов]  {title}')

544 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Фильмы с наибольшим количеством жанров (топ-10):
  1. [6 жанра/ов]  Aladdin and the King of Thieves (1996)
  2. [6 жанра/ов]  All Dogs Go to Heaven 2 (1996)
  3. [6 жанра/ов]  Beauty and the Beast (1991)
  4. [6 жанра/ов]  Getaway, The (1994)
  5. [6 жанра/ов]  Lion King, The (1994)
  6. [6 жанра/ов]  Space Jam (1996)
  7. [6 жанра/ов]  Strange Days (1995)
  8. [6 жанра/ов]  Super Mario Bros. (1993)
  9. [5 жанра/ов]  Aladdin (1992)
 10. [5 жанра/ов]  Alice in Wonderland (1951)


**Вывод:**  
Драма и Комедия лидируют в нашей выборке. Даже на текущий день эти жанры являются наиболее популярными, они же - самые универсальные, а значит дальше именно они будут чаще всплывать и в оценках, и в пользовательских реакциях.

Судя по нашим данным в некоторых фильмах кинематографы не могли чётко определиться с жанром своей работы и внесли по 5-6 жанров в фильм.

## *Вопрос 2: Какие фильмы получают самые высокие оценки? Какие просто собирают больше внимания?*

MovieLens оценивает фильмы от 0,5 до 5,0 звёзд.  

Среднее и медиана — это не одно и то же.
Один восторженный фанат может поднять среднее, но не сдвинет медиану. 

`Среднее` – Сумма всех элементов, поделенная на количество
элементов в числовом ряду.   
`Медиана` – Средний элемент упорядоченного числового ряда   

In [11]:
%timeit -n 1 -r 1 ratings.top_by_ratings(10)

top_avg = ratings.top_by_ratings(10)
print('Фильмы по среднему баллу (топ-10):')
for rank, (title, score) in enumerate(top_avg.items(), 1):
    print(f' {rank:>2}. {score:.2f}   {title}')

605 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Фильмы по среднему баллу (топ-10):
  1. 5.00   12 Angry Men (1957)
  2. 5.00   Adventures of Robin Hood, The (1938)
  3. 5.00   Alice in Wonderland (1951)
  4. 5.00   Aristocats, The (1970)
  5. 5.00   Back to the Future (1985)
  6. 5.00   Basic Instinct (1992)
  7. 5.00   Beautiful Thing (1996)
  8. 5.00   Bedknobs and Broomsticks (1971)
  9. 5.00   Before Sunrise (1995)
 10. 5.00   Bottle Rocket (1996)


In [12]:
%timeit -n 1 -r 1 ratings.top_by_ratings(10, 'median')

top_median = ratings.top_by_ratings(10, 'median')
print('Фильмы по медианному баллу (топ-10):')
for rank, (title, score) in enumerate(top_median.items(), 1):
    print(f' {rank:>2}. {score:.2f}   {title}')

548 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Фильмы по медианному баллу (топ-10):
  1. 5.00   12 Angry Men (1957)
  2. 5.00   Adventures of Robin Hood, The (1938)
  3. 5.00   Alice in Wonderland (1951)
  4. 5.00   Aristocats, The (1970)
  5. 5.00   Back to the Future (1985)
  6. 5.00   Basic Instinct (1992)
  7. 5.00   Beautiful Thing (1996)
  8. 5.00   Bedknobs and Broomsticks (1971)
  9. 5.00   Before Sunrise (1995)
 10. 5.00   Bottle Rocket (1996)


Отвечая на вопрос *"Какие фильмы просто собирают больше внимания?"* даётся следующий результат, без среднего, без медианы, "просто" собирают больше внимания.

In [13]:
%timeit -n 1 -r 1 ratings.top_by_num_of_ratings(10)

top_by_count = ratings.top_by_num_of_ratings(10)
print('Самые часто оцениваемые фильмы (топ-10):')
for rank, (title, cnt) in enumerate(top_by_count.items(), 1):
    print(f'  {rank:>2}. {cnt:>3} оценок — {title}')

283 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Самые часто оцениваемые фильмы (топ-10):
   1.   4 оценок — Aladdin (1992)
   2.   4 оценок — Batman (1989)
   3.   4 оценок — Beauty and the Beast (1991)
   4.   4 оценок — Fargo (1996)
   5.   4 оценок — Fugitive, The (1993)
   6.   4 оценок — Pulp Fiction (1994)
   7.   4 оценок — Schindler's List (1993)
   8.   4 оценок — Silence of the Lambs, The (1991)
   9.   4 оценок — Usual Suspects, The (1995)
  10.   3 оценок — Apollo 13 (1995)


In [14]:
%timeit -n 1 -r 1 ratings.dist_by_year()

dist = ratings.dist_by_year()
print('Распределение оценок по годам:')
max_count = list(dist.values())[0]
for year, cnt in dist.items():
    bar = '█' * int(cnt / max_count * 30)
    print(f' {year}  {bar} {cnt}')

83.5 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Распределение оценок по годам:
 1996  ██████████████████████████████ 358
 1999  ███ 47
 2000  ██████████ 122
 2001  █ 16
 2005  ██ 31
 2011  █ 12
 2015   2


**Вывод:**  
Среднее и Медиана дают нам разные выводы.  
Фильм с 1-2 идеальными оценками может с лёгкостью подняться в топ по Среднему, но Медиана показывает более "честный" результат, так как она не чувствительна к выбросам.  
Распределение оценок по годам показывает, что активность пользователей по времени тоже неравномерна.

## *Вопрос 3: Кто все эти люди, которые оценивают фильмы?*

Теперь интересно посмотреть уже не на фильмы, а на самих пользователей.  
За каждой оценкой стоит человек: кто активнее всех и кто выставляет самые высокие оценки даже на слабые и посредственные фильмы?  

Здесь `class Users` наследуется от `Movies`, который показывает как оценки так и фильмы вместе.  
Пользователей мы выделяем по их ID, поэтому они будут именоваться как числовые "User ID №1"

In [15]:
%timeit -n 1 -r 1 users.most_active_users(10)

active = users.most_active_users(10)
print('Самые активные пользователи (топ-10):')
for rank, (user_id, cnt) in enumerate(active.items(), 1):
    bar = '🙋🏼‍♀️' * (cnt // 20)
    print(f'  {rank:>2}. User ID №{user_id:<4}  {bar} ({cnt} оценок)')

63.8 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Самые активные пользователи (топ-10):
   1. User ID №6     🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️ (314 оценок)
   2. User ID №4     🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️ (94 оценок)
   3. User ID №1     🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️🙋🏼‍♀️ (91 оценок)
   4. User ID №5     🙋🏼‍♀️🙋🏼‍♀️ (44 оценок)
   5. User ID №7     🙋🏼‍♀️ (31 оценок)
   6. User ID №3      (12 оценок)
   7. User ID №2      (2 оценок)


In [16]:
%timeit -n 1 -r 1 users.most_generous_users(10)

generous = users.most_generous_users(10)
print('Кто из пользователей оценивает фильмы по высокому баллу (топ-10):')
for rank, (user_id, avg) in enumerate(generous.items(), 1):
    print(f'  {rank:>2}. User ID №{user_id:<4} Средняя оценка = {avg:.2f}')

77.2 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Кто из пользователей оценивает фильмы по высокому баллу (топ-10):
   1. User ID №1    Средняя оценка = 4.41
   2. User ID №7    Средняя оценка = 3.89
   3. User ID №4    Средняя оценка = 3.72
   4. User ID №5    Средняя оценка = 3.64
   5. User ID №2    Средняя оценка = 3.50
   6. User ID №6    Средняя оценка = 3.49
   7. User ID №3    Средняя оценка = 1.12


**Вывод:**  
По выведенным данным можно сказать, что активность и щедрость абсолютно не связаны друг с другом.  
Например, *User ID №6* поставил 314 оценок, он очень активный пользователь, но его средняя оценка составляет <span style="color:red">3,49</span>.  

Один пользователь может ставить очень много оценок, а другой — просто быть мягче в суждениях.  
Возможно *User ID №1*, у которого третье место по активности оценивания фильмов, смотрит только те фильмы, которые ему точно понравятся и поэтому его средний балл составляет <span style="color:yellow">4,41</span>.

## *Вопрос 4: Язык пользователей – #Тег*

Оценка – это число, а Тег – это слово. Теги интересны тем, что это уже почти живая речь. В них меньше формальности и больше интуитивной реакции.

In [17]:
%timeit -n 1 -r 1 tags.most_words(10)

wordy = tags.most_words(10)
print('Теги с наибольшим числом слов (топ-10):')
for tag_text, word_cnt in wordy.items():
    print(f'  [{word_cnt} слов]  {tag_text!r}')

574 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Теги с наибольшим числом слов (топ-10):
  [16 слов]  'Something for everyone in this one... saw it without and plan on seeing it with kids!'
  [10 слов]  'the catholic church is the most corrupt organization in history'
  [6 слов]  'Oscar (Best Music - Original Score)'
  [5 слов]  'Everything you want is here'
  [5 слов]  'based on a true story'
  [4 слов]  'Guardians of the Galaxy'
  [4 слов]  'a clever chef rat'
  [4 слов]  'based on a book'
  [4 слов]  'based on a play'
  [4 слов]  'end of the world'


Здесь мы можем видеть, что пользователи ищут фильм без выбора, на "просто посмотреть".  
Некоторые довольно придирчивы и специфичны – ищут громкие фильмы, которые являются номинантом или лауреатом Оскара.  
Другие же ищут некий байопик "основано на реальных событиях" или совсем отчаялись... ищут "конец света".

In [18]:
%timeit -n 1 -r 1 tags.longest(10)
long_tags = tags.longest(10)

print('Самые длинные теги по числу символов (топ-10):')
for tag_text in long_tags:
    print(f'  [{len(tag_text)} симв.]  {tag_text!r}')

394 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Самые длинные теги по числу символов (топ-10):
  [85 симв.]  'Something for everyone in this one... saw it without and plan on seeing it with kids!'
  [63 симв.]  'the catholic church is the most corrupt organization in history'
  [36 симв.]  'audience intelligence underestimated'
  [35 симв.]  'Oscar (Best Music - Original Score)'
  [28 симв.]  'assassin-in-training (scene)'
  [27 симв.]  'Everything you want is here'
  [27 симв.]  'Oscar (Best Cinematography)'
  [27 симв.]  'political right versus left'
  [26 симв.]  'representation of children'
  [23 симв.]  'Disney animated feature'


In [19]:
%timeit -n 1 -r 1 tags.most_popular(15)

popular = tags.most_popular(15)
print('Самые популярные Теги (топ-15):')
max_cnt = list(popular.values())[0]
for tag_name, cnt in popular.items():
    bar = '█' * int(cnt / max_cnt * 20)
    print(f'  {tag_name:<34} {bar} ({cnt})')

528 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Самые популярные Теги (топ-15):
  funny                              ████████████████████ (15)
  sci-fi                             ██████████████████ (14)
  dark comedy                        ████████████████ (12)
  twist ending                       ████████████████ (12)
  action                             █████████████ (10)
  atmospheric                        █████████████ (10)
  comedy                             █████████████ (10)
  superhero                          █████████████ (10)
  suspense                           █████████████ (10)
  Leonardo DiCaprio                  ████████████ (9)
  comic book                         ████████████ (9)
  psychology                         ████████████ (9)
  satire                             ████████████ (9)
  thought-provoking                  ██████████ (8)
  mindfuck                           █████████ (7)


Просмотр Тегов довольно интересное занятие. Большинство пользователей ищут смешные комедии или научную фантастику.  
Меньше всего Тегов на просмотр таких жанров как психология, провокация или даже *Майндфак*, который может "выносить мозг" со своей запутанностью сюжета.

In [20]:
%timeit -n 1 -r 1 tags.tags_with('sci')
print('Теги, содержащие слово "sci":')
for t in tags.tags_with('sci'):
    print(f'  — {t}')
print('-' * 80)

%timeit -n 1 -r 1 tags.tags_with('love')
print('Теги, содержащие слово "love":')
for t in tags.tags_with('love'):
    print(f'  — {t}')
print('-' * 80)

%timeit -n 1 -r 1 tags.tags_with('classic')
print('Теги, содержащие слово "classic":')
for t in tags.tags_with('classic'):
    print(f'  — {t}')
print('-' * 80)

%timeit -n 1 -r 1 tags.tags_with('space')
print('Теги, содержащие слово "space":')
for t in tags.tags_with('space'):
    print(f'  — {t}')

133 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Теги, содержащие слово "sci":
  — Sci-Fi
  — classic sci-fi
  — mad scientist
  — sci-fi
  — science fiction
  — scifi
  — scifi cult
  — sexy female scientist
--------------------------------------------------------------------------------
96.2 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Теги, содержащие слово "love":
  — love story
--------------------------------------------------------------------------------
74 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Теги, содержащие слово "classic":
  — Classic
  — classic
  — classic sci-fi
  — updated classics
--------------------------------------------------------------------------------
77 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Теги, содержащие слово "space":
  — SPACE TRAVEL
  — space
  — space action
  — space adventure
  — space craft
  — space epic


**Вывод:**  
Получается любопытный контраст: самые частые теги короткие и удобные, а самые длинные больше похожи на мини-комментарии. это уже не просто метка, а почти маленькая рецензия.

## *Вопрос 5: HTTP-запросы к IMDB. Что можно взять из внешних ссылок?*

**Данная часть делает HTTP-запросы к сайту [IMBD](https://www.imdb.com/?ref_=test).**  

⚠️ При ошибке запроса, модуль возвращает <span style="color:red">Exception</span> `request failed`. Обработка *уже* встроена в метод `get_imbd()`.

In [21]:
%timeit -n 1 -r 1 links.get_imdb_link('Toy Story (1995)')

title = 'Toy Story (1995)'
print('IMDB-ссылка для фильма из датасета:')
link = links.get_imdb_link(title)
print(f'  {title}  →   {link}')

3.46 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
IMDB-ссылка для фильма из датасета:
  Toy Story (1995)  →   https://www.imdb.com/title/tt0114709/


In [22]:
%timeit -n 1 -r 1 links.get_imdb('Batman Forever (1995)'),

result = links.get_imdb('Batman Forever (1995)')
print('Данные IMDB для «Batman Forever (1995)»:')
print(f"  Статус запроса : {result['status']}")
print('-' * 80)
print(f"  Название фильма: {result['movie_title']}")
print(f"  IMDB ID        : {result['imdb_id']}")
print(f"  Ссылка         : {result['imdb_link']}")

314 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Данные IMDB для «Batman Forever (1995)»:
  Статус запроса : ok
--------------------------------------------------------------------------------
  Название фильма: Batman Forever (1995)
  IMDB ID        : 0112462
  Ссылка         : https://www.imdb.com/title/tt0112462/


# ⭐ **Бонусная часть**

В бонусной части мы используем два дополнительных метода, которые расширяют аналитику:

1. **`movies_with_genre(genre, n)`** — фильтрует фильмы по конкретному жанру и возвращает первые `n` результатов в алфавитном порядке. Это позволяет нам быстро получить выборку по интересующему жанру.
2. **`links.missing_tmdb(n)`** — находит фильмы, у которых отсутствует TMDB ID. Если у фильма нет ссылки на TMDB, значит он может быть засчитан как выброс и не будет полезным для анализа.

In [23]:
%timeit -n 1 -r 1 movies.movies_with_genre('Drama', 10)

drama_films = movies.movies_with_genre('Drama', 10)
print('Первые 10 драм из датасета:')
for title in drama_films:
    print(f' – {title}')

234 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Первые 10 драм из датасета:
 – 'Til There Was You (1997)
 – 1-900 (06) (1994)
 – 12 Angry Men (1957)
 – 20,000 Leagues Under the Sea (1954)
 – 2001: A Space Odyssey (1968)
 – 39 Steps, The (1935)
 – 8 1/2 (8½) (1963)
 – 8 Seconds (1994)
 – Above the Rim (1994)
 – Addiction, The (1995)


Теперь проверим целостность данных — у всех ли фильмов есть ссылка на TMDB (The Movie Database).  
Фильмы без TMDB ID — это "слепые пятна" датасета, которые невозможно сопоставить с внешними источниками.

In [24]:
%timeit -n 1 -r 1 links.missing_tmdb(10)

missing = links.missing_tmdb(10)
print(f'Фильмы без TMDB ID (найдено {len(missing)}):')
for title in missing:
    print(f' ❌ {title}')

71.4 µs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Фильмы без TMDB ID (найдено 2):
 ❌ Last Klezmer: Leopold Kozlowski, His Life and Music, The (1994)
 ❌ Loser (1991)


**Вывод по бонусной части:**

- **Фильтрация по жанру** (`movies_with_genre`) подтверждает доминирование Драмы в датасете: даже первые 10 результатов охватывают широкий временной диапазон — от классики 1935 года до фильмов середины 90-х. Это говорит о том, что жанр "Драма" универсален и не привязан к конкретной эпохе.

- **Проверка TMDB-ссылок** (`missing_tmdb`) выявила всего 2 фильма без TMDB ID (*но из данных нам 1000 строк*): <span style="color:red">Last Klezmer (1994)</span> и <span style="color:red">Loser (1991)</span>.  
Оба — малоизвестные нишевые картины. Это означает, что покрытие данных TMDB составляет ~99.8%, что является отличным показателем полноты датасета. 

# 🎬 **Заключение**

Даже на первых 1000 строках видно несколько слоев одной истории. Мы начали с четырёх CSV-файлов. Что мы нашли?:

- **Драма и Комедия – превыше всего**: не потому что они лучшие, а потому что они самые понятные зрителю.
- **Разное оценивание фильмов**: зрители выбирают смотреть то, что им скорее понравится. Пессимизм редко встречается. По рейтингам видно различие между качеством по оценке и простым вниманием аудитории.  
- **Небольшая группа пользователей создаёт основной поток данных**: по пользователям заметно, что активность и щедрость — разные роли.
- **Теги как живой голос**: короткие, точные, эмоционально заряженные. Числа говорят «что», а теги «почему». Слышен живой пользовательский язык.
- **IMDB как зеркало**: то, что любит сообщество MovieLens, не всегда совпадает с мнением миллионов. По ссылкам видно, что модуль умеет аккуратно работать даже там, где может подвести сеть.

Данный модуль не только считает, но и помогает рассказать маленькую историю о данных.  

*Отчёт создан с использованием `movielens_analysis.py`. Все методы использованы. Каждая ячейка замерена через `%timeit`.*  